# Nhóm 13 — M1: Crawl và benchmark toàn bộ nhóm
- **Mục tiêu:** Thu thập dữ liệu bài báo arXiv và trích dẫn Semantic Scholar cho Milestone 1.
- **Môi trường bắt buộc:** Colab **CPU** (không chọn GPU/TPU để đảm bảo benchmark chuẩn xác).
- **Secrets cần nạp:** Bật *Notebook access* cho `ARXIV_CONTACT_EMAIL` và `SEMANTIC_SCHOLAR_API_KEY`.
- **Đồng bộ Google Drive:** Dữ liệu và nhật ký được ghi trực tiếp vào thư mục `IDS_Nhom13_Corpus_Backup`.
- **Hỗ trợ cào song song:** Các thành viên có thể chạy worker với MSSV cá nhân (`23120364`, `23120358`, `23120345`) mà không xung đột dữ liệu.


In [1]:
# @title Setup: Đọc Secrets và Mount Google Drive
import os
from google.colab import userdata, drive

for name in ('ARXIV_CONTACT_EMAIL', 'SEMANTIC_SCHOLAR_API_KEY'):
    os.environ[name] = userdata.get(name)
print('Đã đọc hai Secrets; không hiển thị giá trị.')

drive.mount('/content/drive')


Đã đọc hai Secrets; không hiển thị giá trị.
Mounted at /content/drive


In [2]:
# @title Source: Xác minh Checksum và Giải nén Gói Mã Nguồn
from pathlib import Path
import hashlib, zipfile

CODE_ROOT = None
SOURCE_SHA256 = '28c10a118fd96e0b63f39ed11d528274928ac6c188b818dae07263e60d5a2983'
DRIVE_ROOT = Path('/content/drive/MyDrive/IDS_Nhom13_Corpus_Backup')
assert DRIVE_ROOT.is_dir(), 'Không thấy thư mục Drive: /content/drive/MyDrive/IDS_Nhom13_Corpus_Backup'

archive = DRIVE_ROOT / ('ids13-code-' + SOURCE_SHA256[:12] + '.zip')
expected = SOURCE_SHA256
assert archive.exists(), f'Không tìm thấy file bundle: {archive}\nHãy tải file {archive.name} vào Drive.'
assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, 'Source checksum mismatch'

CODE_ROOT = Path('/content') / ('ids13-code-' + expected[:12])
CODE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for name in bundle.namelist():
        assert (CODE_ROOT / name).resolve().is_relative_to(CODE_ROOT.resolve()), f'Path traversal: {name}'
    bundle.extractall(CODE_ROOT)

print('✅ Source verified:', expected[:12])
print('   Thư mục mã nguồn:', CODE_ROOT)


✅ Source verified: 28c10a118fd9
   Thư mục mã nguồn: /content/ids13-code-28c10a118fd9


## Khởi chạy Worker
Hệ thống hỗ trợ 2 chế độ vận hành:
1. **Chế độ đơn (Single worker):** Cào toàn bộ 15.000 bài (để `STUDENT_ID = None`).
2. **Chế độ cào song song 3 thành viên:** Chọn MSSV tương ứng để cào đúng dải được phân công:
   - **Phạm Minh Thông (23120364):** Dải `2508.17872` → `2509.01046`
   - **Nguyễn Tiến Thành (23120358):** Dải `2509.01047` → `2509.06046`
   - **Nguyễn Trịnh Quang (23120345):** Dải `2509.06047` → `2509.11046`

Dữ liệu được lưu độc lập theo mã bài (`m1-final-v3/raw/yymm-id/`), không ghi đè dữ liệu của nhau.


In [3]:
# @title Khởi động Worker Crawler (Chạy nền)
import subprocess, sys, json, os
from pathlib import Path

assert CODE_ROOT is not None and CODE_ROOT.is_dir(), 'Hãy chạy cell source thành công trước!'
assert CODE_ROOT.name.endswith(SOURCE_SHA256[:12]), 'Phiên bản mã nguồn không khớp!'
os.environ['IDS_SOURCE_SHA256'] = SOURCE_SHA256

# Chọn MSSV của bạn (hoặc None nếu Thông chạy toàn bộ dải):
STUDENT_ID = None  # @param ["None", "23120364", "23120358", "23120345"] {allow-input: true}
if STUDENT_ID in ("None", "", None):
    STUDENT_ID = None

suffix = f"-{STUDENT_ID}" if STUDENT_ID else ""
status_path = DRIVE_ROOT / f"m1-launch-status{suffix}.json"
log_path = DRIVE_ROOT / f"m1-agent{suffix}.log"

# Kiểm tra nếu worker cũ vẫn đang chạy trên máy này
if status_path.exists():
    previous = json.loads(status_path.read_text())
    previous_pid = previous.get('pid')
    if previous_pid:
        command_file = Path('/proc') / str(previous_pid) / 'cmdline'
        if command_file.exists() and b'colab_install_and_run.py' in command_file.read_bytes():
            raise RuntimeError(f'Worker (PID {previous_pid}) đang chạy! Không khởi chạy trùng lặp.')

for name in ('ARXIV_CONTACT_EMAIL', 'SEMANTIC_SCHOLAR_API_KEY'):
    assert os.environ.get(name), 'Secret chưa được nạp: ' + name

cmd = [sys.executable, str(CODE_ROOT / 'scripts/colab_install_and_run.py'), '--root', str(DRIVE_ROOT)]
if STUDENT_ID:
    cmd.extend(['--student-id', STUDENT_ID, '--log-suffix', STUDENT_ID])

log_handle = log_path.open('a', buffering=1)
worker = subprocess.Popen(cmd, cwd=CODE_ROOT, stdout=log_handle, stderr=subprocess.STDOUT, env=os.environ.copy())
print('✅ Worker đã khởi động thành công!')
print(f'   PID: {worker.pid}')
print(f'   Phân công: {STUDENT_ID or "Toàn bộ nhóm (Single worker)"}')
print(f'   File log: {log_path}')
print(f'   File trạng thái: {status_path}')


✅ Worker đã khởi động thành công!
   PID: 2138
   Phân công: Toàn bộ nhóm (Single worker)
   File log: /content/drive/MyDrive/IDS_Nhom13_Corpus_Backup/m1-agent.log
   File trạng thái: /content/drive/MyDrive/IDS_Nhom13_Corpus_Backup/m1-launch-status.json


In [4]:
# @title Giám sát Tiến độ và Xem Nhật ký (Monitor)
import json
from pathlib import Path

status_files = [
    'm1-launch-status.json',
    'm1-launch-status-23120364.json',
    'm1-launch-status-23120358.json',
    'm1-launch-status-23120345.json',
    'm1-automation-status.json',
    'm1-final-v3/raw/crawl_summary.json',
    'm1-pilot-v3/raw/crawl_summary.json'
]

print('📊 TRẠNG THÁI HIỆN TẠI TRÊN DRIVE:')
for filename in status_files:
    path = DRIVE_ROOT / filename
    if path.exists():
        value = json.loads(path.read_text())
        value.pop('failed_ids', None)
        print(f'  • {filename}: {json.dumps(value, ensure_ascii=False)}')

for log_file in sorted(DRIVE_ROOT.glob('m1-agent*.log')):
    print(f'\n📝 15 DÒNG NHẬT KÝ GẦN NHẤT ({log_file.name}):')
    tail = log_file.read_text(errors='replace').splitlines()[-15:]
    print('\n'.join(tail))


In [5]:
# @title Kiểm tra Chi tiết các bài cào (10 bài mới nhất)
import json

root = DRIVE_ROOT / 'm1-final-v3/raw'
if not root.exists() or not any(root.glob('*/crawl_status.json')):
    root = DRIVE_ROOT / 'm1-pilot-v3/raw'

statuses = sorted(root.glob('*/crawl_status.json'))
print(f'Tổng số bài đã ghi nhận crawl_status: {len(statuses)}')
for path in statuses[-10:]:
    r = json.loads(path.read_text())
    print(json.dumps({
        'id': r['arxiv_id'],
        'success': r['success'],
        'attempts': r.get('attempt_count'),
        'error': r.get('error'),
        'versions': [{k: v.get(k) for k in ('version', 'success', 'state', 'error', 'downloaded_bytes')} for v in r.get('versions', [])],
        'references': r.get('references_status')
    }, ensure_ascii=False))


In [6]:
# @title Kiểm tra Tiến trình Worker Đang chạy
import json, psutil

for sf in sorted(DRIVE_ROOT.glob('m1-launch-status*.json')):
    try:
        launch = json.loads(sf.read_text())
        pid = launch.get('pid')
        alive = psutil.pid_exists(pid) if pid else False
        print(f'{sf.name}: PID={pid}, Alive={alive}')
    except Exception as e:
        print(f'{sf.name}: lỗi đọc - {e}')
